# **Working with JSON Data**

![](../images/JSON_object.png)

In [1]:
import json

In [2]:
sample_json = """{
    "id": 143,
    "name": "Silicon Valley",
    "type": "Scripted",
    "language": "English",
    "genres": [
        "Comedy"
    ],
    "network": {
        "id": 8,
        "name": "HBO",
        "country": {
        "name": "United States",
        "code": "US",
        "timezone": "America/New_York"
        }
    }
}"""

In [3]:
document = json.loads(sample_json)

document

{'id': 143,
 'name': 'Silicon Valley',
 'type': 'Scripted',
 'language': 'English',
 'genres': ['Comedy'],
 'network': {'id': 8,
  'name': 'HBO',
  'country': {'name': 'United States',
   'code': 'US',
   'timezone': 'America/New_York'}}}

In [4]:
type(document)

dict

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.utils import AnalysisException

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/14 14:41:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=pyspark-shell>

In [7]:
document = spark.read.json('../data/shows/shows-silicon-valley.json')

document.count()

1

In [8]:
documents = spark.read.json(
    '../data/shows/shows-*.json', 
    multiLine=True)

documents.count()

26/11/13 11:14:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/shows/shows-*.json.
java.io.FileNotFoundException: File ../data/shows/shows-*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.an

3

In [9]:
documents.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [10]:
documents.columns

['_embedded',
 '_links',
 'externals',
 'genres',
 'id',
 'image',
 'language',
 'name',
 'network',
 'officialSite',
 'premiered',
 'rating',
 'runtime',
 'schedule',
 'status',
 'summary',
 'type',
 'updated',
 'url',
 'webChannel',
 'weight']

## **The Array Type**

In [11]:
array_subset = document.select('name', 'genres')

array_subset.show(1, False)

+--------------+--------+
|name          |genres  |
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



![](../images/array_subset.png)

In [12]:
array_subset = array_subset.select(
    'name',
    array_subset.genres[0].alias('dot_and_index'),
    F.col('genres')[0].alias('col_and_index'),
    array_subset.genres.getItem(0).alias('dot_and_method'),
    F.col('genres').getItem(0).alias('col_and_method')
)

array_subset.show(truncate=False)

+--------------+-------------+-------------+--------------+--------------+
|name          |dot_and_index|col_and_index|dot_and_method|col_and_method|
+--------------+-------------+-------------+--------------+--------------+
|Silicon Valley|Comedy       |Comedy       |Comedy        |Comedy        |
+--------------+-------------+-------------+--------------+--------------+



In [13]:
array_subset_repeated = array_subset.select(
    'name',
    F.lit('Comedy').alias('one'),
    F.lit('Horror').alias('two'),
    F.lit('Drama').alias('three'),
    F.col('dot_and_index')
)

array_subset_repeated.show()

+--------------+------+------+-----+-------------+
|          name|   one|   two|three|dot_and_index|
+--------------+------+------+-----+-------------+
|Silicon Valley|Comedy|Horror|Drama|       Comedy|
+--------------+------+------+-----+-------------+



In [14]:
array_subset_repeated = array_subset_repeated.select(
    'name',
    F.array('one', 'two', 'three').alias('Some_Genres'),
    F.array_repeat('dot_and_index', 5).alias('Repeated_Genres')
)

array_subset_repeated.show(truncate=False)

+--------------+-----------------------+----------------------------------------+
|name          |Some_Genres            |Repeated_Genres                         |
+--------------+-----------------------+----------------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]|[Comedy, Comedy, Comedy, Comedy, Comedy]|
+--------------+-----------------------+----------------------------------------+



In [15]:
array_subset_repeated.select(
    'name',
    F.size('Some_Genres'),
    F.size('Repeated_Genres')
).show()

+--------------+-----------------+---------------------+
|          name|size(Some_Genres)|size(Repeated_Genres)|
+--------------+-----------------+---------------------+
|Silicon Valley|                3|                    5|
+--------------+-----------------+---------------------+



In [16]:
array_subset_repeated.select(
    'name',
    F.array_distinct('Some_Genres'),
    F.array_distinct('Repeated_Genres')
).show(1, truncate=False)

+--------------+---------------------------+-------------------------------+
|name          |array_distinct(Some_Genres)|array_distinct(Repeated_Genres)|
+--------------+---------------------------+-------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]    |[Comedy]                       |
+--------------+---------------------------+-------------------------------+



In [17]:
array_subset_repeated = array_subset_repeated.select(
    'name',
    F.array_intersect('Some_Genres', 'Repeated_Genres').alias('Genres')
)

array_subset_repeated.show()

+--------------+--------+
|          name|  Genres|
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



In [18]:
array_subset_repeated.select(
    'Genres',
    F.array_position('Genres', 'Comedy')
).show()

+--------+------------------------------+
|  Genres|array_position(Genres, Comedy)|
+--------+------------------------------+
|[Comedy]|                             1|
+--------+------------------------------+



## **The Map Type**

In [19]:
columns = ["name", "language", "type"]

df_map = documents.select(
    *[F.lit(col) for col in columns],
    F.array(*columns).alias('values')
)

df_map.show(3, False)

+----+--------+----+-------------------------------------+
|name|language|type|values                               |
+----+--------+----+-------------------------------------+
|name|language|type|[The Golden Girls, English, Scripted]|
|name|language|type|[Breaking Bad, English, Scripted]    |
|name|language|type|[Silicon Valley, English, Scripted]  |
+----+--------+----+-------------------------------------+



In [20]:
df_map = df_map.select(
    F.array(*columns).alias('keys'), 'values'
)

df_map.show(3, False)

+----------------------+-------------------------------------+
|keys                  |values                               |
+----------------------+-------------------------------------+
|[name, language, type]|[The Golden Girls, English, Scripted]|
|[name, language, type]|[Breaking Bad, English, Scripted]    |
|[name, language, type]|[Silicon Valley, English, Scripted]  |
+----------------------+-------------------------------------+



In [21]:
df_map = df_map.select(
    F.map_from_arrays('keys', 'values').alias('mapped')
)

df_map.printSchema()

root
 |-- mapped: map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [22]:
df_map.show(3, False)

+-----------------------------------------------------------------+
|mapped                                                           |
+-----------------------------------------------------------------+
|{name -> The Golden Girls, language -> English, type -> Scripted}|
|{name -> Breaking Bad, language -> English, type -> Scripted}    |
|{name -> Silicon Valley, language -> English, type -> Scripted}  |
+-----------------------------------------------------------------+



In [23]:
df_map.select(
    F.col('mapped.name'),
    F.col('mapped')['name'],
    df_map.mapped['name']
).show()

+----------------+----------------+----------------+
|            name|    mapped[name]|    mapped[name]|
+----------------+----------------+----------------+
|The Golden Girls|The Golden Girls|The Golden Girls|
|    Breaking Bad|    Breaking Bad|    Breaking Bad|
|  Silicon Valley|  Silicon Valley|  Silicon Valley|
+----------------+----------------+----------------+



## **The Struct Type**

In [24]:
documents.select('schedule').printSchema()

root
 |-- schedule: struct (nullable = true)
 |    |-- days: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- time: string (nullable = true)



![](../images/struct_vis.png)

In [25]:
documents.select(F.col("_embedded")).printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

![](../images/_embedded_field.png)

In [26]:
documents_clean = documents.withColumn(
    'episodes', F.col('_embedded.episodes')
).drop('_embedded')

documents_clean.printSchema()

root
 |-- _links: struct (nullable = true)
 |    |-- previousepisode: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- self: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |-- externals: struct (nullable = true)
 |    |-- imdb: string (nullable = true)
 |    |-- thetvdb: long (nullable = true)
 |    |-- tvrage: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- medium: string (nullable = true)
 |    |-- original: string (nullable = true)
 |-- language: string (nullable = true)
 |-- name: string (nullable = true)
 |-- network: struct (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- timezone: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nul

In [27]:
episodes_name = documents_clean.select(F.col('episodes.name'))

episodes_name.printSchema()

root
 |-- name: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [28]:
episodes_name.select(
    F.explode('name').alias('name')
).show(3, False)

+----------------------------------+
|name                              |
+----------------------------------+
|The Engagement                    |
|Guess Who's Coming to the Wedding?|
|Rose the Prude                    |
+----------------------------------+
only showing top 3 rows


In Apache Spark, **Complex Data Types** (`array`, `map`, and `struct`) allow you to handle semi-structured, nested, or multi-valued data directly inside tabular DataFrames without needing to flatten everything into rigid, flat flat-file structures.

Under Project Tungsten and the Catalyst Optimizer, these complex types are stored compactly in binary off-heap memory formats, minimizing Java Garbage Collection (GC) overhead while providing rich SQL-like manipulation functions.

---

## **1. Array (`ArrayType`)**

An **Array** is an ordered collection of elements where **all elements must share the exact same data type** (e.g., an array of strings, an array of integers).

### **⚙️ Characteristics**

* **Homogeneous:** Every item in the array must be of the same type.
* **Ordered:** Elements maintain their insertion or specified sequence, accessible via 1-based indexing (e.g., `element_at(col, 1)`) or 0-based indexing depending on the specific function.
* **Nullability:** Arrays can contain `null` elements, and the array itself can be null.
* **Duplicates:** Duplicate values are allowed.

### **🛠️ Common Usage & PySpark Example**

Arrays are ideal for storing multi-valued attributes like tags, lists of product IDs, clickstream path histories, or tokenized words.

```python
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ComplexTypesExample").getOrCreate()

# Sample data with an array of tags and scores
data = [
    (1, "Alice", ["python", "spark", "sql"]),
    (2, "Bob", ["java", "spring"]),
]
df = spark.createDataFrame(data, ["id", "name", "skills"])

# 1. Check if array contains a specific value
df_filtered = df.filter(F.array_contains(F.col("skills"), "spark"))

# 2. Explode the array into individual rows (one row per skill)
df_exploded = df.select("id", "name", F.explode("skills").alias("skill"))
df_exploded.show(truncate=False)

```

---

## **2. Map (`MapType`)**

A **Map** is a collection of key-value pairs (dictionary structure). **All keys must share the same data type**, and **all values must share the same data type**.

### **⚙️ Characteristics**

* **Key-Value Pair Structure:** Data is looked up by key rather than index position.
* **Homogeneous Types:** All keys share a common type (frequently `StringType`), and all values share a common type.
* **Key Constraints:** Keys **cannot be null**. Values can be null.
* **Unordered:** Maps do not guarantee an ordered sequence of elements.

### **🛠️ Common Usage & PySpark Example**

Maps are powerful for dynamic attributes, flexible telemetry metadata, configuration settings, or multi-language translation dictionaries where keys vary per row.

```python
# Sample data with a map of metadata attributes
data = [
    (1, {"env": "prod", "version": "v1.2", "region": "us-east"}),
    (2, {"env": "dev", "version": "v2.0"}),
]
df = spark.createDataFrame(data, ["id", "metadata"])

# 1. Access a specific map value by key using bracket notation or element_at
df_env = df.select("id", df.metadata["env"].alias("environment"))

# 2. Extract all keys or values as arrays
df_keys = df.select("id", F.map_keys("metadata").alias("all_keys"))

df_env.show(truncate=False)

```

---

## **3. Struct (`StructType`)**

A **Struct** is a record type containing a sequence of named fields with associated data types. It is structurally identical to a row in a table, a JSON object, or a nested Python dictionary.

### **⚙️ Characteristics**

* **Heterogeneous Fields:** Unlike arrays and maps, fields within a struct **can have completely different data types** (e.g., `Field1` is an integer, `Field2` is a string, `Field3` is an array).
* **Named Access:** Fields are accessed by name (dot notation like `struct_col.field_name`) or position.
* **Building Block of Nesting:** Structs form the backbone of nested DataFrames when reading JSON, Parquet, or Delta Lake formats with hierarchical schemas.

### **🛠️ Common Usage & PySpark Example**

Structs are used to group related attributes together—such as bundling `street`, `city`, and `zipcode` into an `address` struct column.

```python
# Sample data with a nested struct for address
data = [
    (1, "Alice", {"city": "Paris", "zip": "75001"}),
    (2, "Bob", {"city": "Berlin", "zip": "10115"}),
]
# Schema is automatically inferred as StructType containing 'city' and 'zip'
df = spark.createDataFrame(data, ["id", "name", "address"])

# 1. Access nested fields using dot notation
df_city = df.select("name", F.col("address.city").alias("city"))

# 2. Flatten/expand all fields of a struct using the star selector (*)
df_unpacked = df.select("id", "name", "address.*")

df_unpacked.show(truncate=False)

```

---

### **📊 Summary Comparison Matrix**

| Feature | **Array (`ArrayType`)** | **Map (`MapType`)** | **Struct (`StructType`)** |
| --- | --- | --- | --- |
| **Data Structure** | Ordered list of items | Key-value pairs dictionary | Collection of named fields |
| **Data Types** | **Homogeneous** (All items same type) | **Homogeneous keys**, **Homogeneous values** | **Heterogeneous** (Fields can have unique types) |
| **Null Keys Allowed?** | N/A (Index-based) | **No** (Keys cannot be null) | N/A (Field names are static strings) |
| **Access Pattern** | By index position or iteration | By key lookup (`map['key']`) | By dot notation (`struct.field`) |
| **Primary Use Case** | Tags, lists, history logs | Flexible attributes, tags, config | Nested entities, JSON records, rows |

---

### **⚠️ Performance & Architecture Considerations**

1. **Explosion Overhead (`explode`):** Using functions like `F.explode()` on arrays or maps performs a lateral join under the hood, duplicating parent row data for every element in the collection. On massive datasets, this can cause a massive data expansion and trigger expensive network shuffles.
2. **Pushdown Limitations:** Traditional storage layers struggle to push down predicate filters inside deeply nested maps or arrays compared to top-level flat columns. When designing performance-critical tables, frequently queried attributes should be kept as flat top-level columns rather than buried inside maps or arrays.

In [50]:
episode_links_schema = T.StructType(
    [
        T.StructField(
            'self', T.StructType([T.StructField('href', T.StringType())])
        )
    ]
)

episode_image_schema = T.StructType(
    [
        T.StructField('meduim', T.StringType()),
        T.StructField('original', T.StringType())
    ]
)

episodes_schema = T.StructType(
    [
        T.StructField('_links', episode_links_schema),
        T.StructField('airdate', T.DateType()),
        T.StructField('airstamp', T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema),
        T.StructField("name", T.StringType()),
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.StringType()),
        T.StructField("url", T.StringType()),
    ]
)

embedded_schema = T.StructType(
    [
        T.StructField(
            '_embedded', T.StructType(
                [
                    T.StructField(
                       'episodes', T.ArrayType(episodes_schema) 
                    )
                ]
            )
        )
    ]
)

In [33]:
documents.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [51]:
document_with_schema = spark.read.json(
    '../data/shows/shows-silicon-valley.json',
    schema=embedded_schema,
    mode='FAILFAST'
)

In [52]:
document_with_schema.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: date (nullable = true)
 |    |    |    |-- airstamp: timestamp (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- meduim: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = t

In [53]:
for col in ['airdate', 'airstamp']:
    document_with_schema.select(
        f"_embedded.episodes.{col}"
    ).select(
        F.explode(col)
    ).show(5)

+----------+
|       col|
+----------+
|2014-04-06|
|2014-04-13|
|2014-04-20|
|2014-04-27|
|2014-05-04|
+----------+
only showing top 5 rows
+-------------------+
|                col|
+-------------------+
|2014-04-07 02:00:00|
|2014-04-14 02:00:00|
|2014-04-21 02:00:00|
|2014-04-28 02:00:00|
|2014-05-05 02:00:00|
+-------------------+
only showing top 5 rows


In [46]:
from py4j.protocol import Py4JJavaError

In [49]:
episodes_schema_BAD = T.StructType(
    [
        T.StructField('_links', episode_links_schema),
        T.StructField('airdate', T.DateType()),
        T.StructField('airstamp', T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema),
        T.StructField("name", T.StringType()),
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.LongType()),
        T.StructField("url", T.LongType()),
    ]
)

embedded_schema2 = T.StructType(
    [
        T.StructField(
            '_embedded', T.StructType(
                [
                    T.StructField(
                       'episodes', T.ArrayType(episodes_schema_BAD) 
                    )
                ]
            )
        )
    ]
)

document_with_schema_wrong = spark.read.json(
    "../data/shows/shows-silicon-valley.json",
    schema=embedded_schema2,
    mode="FAILFAST",
)

try:
    document_with_schema_wrong.show()
except Py4JJavaError:
    pass

26/11/13 12:09:36 ERROR Executor: Exception in task 0.0 in stage 35.0 (TID 39)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///workspace/data/shows/shows-silicon-valley.json.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:940)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:149)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory

In [54]:
import pprint

In [56]:
pprint.pprint(
    document_with_schema.select(
        F.explode("_embedded.episodes").alias('episode')
    )
    .select('episode.airtime')
    .schema.jsonValue()
)

{'fields': [{'metadata': {},
             'name': 'airtime',
             'nullable': True,
             'type': 'string'}],
 'type': 'struct'}


In [57]:
pprint.pprint(
    T.StructField(
        "array_example", 
        T.ArrayType(T.StringType())
    ).jsonValue()
)

{'metadata': {},
 'name': 'array_example',
 'nullable': True,
 'type': {'containsNull': True, 'elementType': 'string', 'type': 'array'}}


In [58]:
pprint.pprint(
    T.StructField(
        "map_example", 
        T.MapType(T.StringType(), T.LongType())
    ).jsonValue()
)

{'metadata': {},
 'name': 'map_example',
 'nullable': True,
 'type': {'keyType': 'string',
          'type': 'map',
          'valueContainsNull': True,
          'valueType': 'long'}}


In [59]:
pprint.pprint(
    T.StructType(
        [
            T.StructField(
                "map_example", 
                T.MapType(T.StringType(), T.LongType())
            ),
            T.StructField(
                "array_example", 
                T.ArrayType(T.StringType())),
        ]).jsonValue()
)

{'fields': [{'metadata': {},
             'name': 'map_example',
             'nullable': True,
             'type': {'keyType': 'string',
                      'type': 'map',
                      'valueContainsNull': True,
                      'valueType': 'long'}},
            {'metadata': {},
             'name': 'array_example',
             'nullable': True,
             'type': {'containsNull': True,
                      'elementType': 'string',
                      'type': 'array'}}],
 'type': 'struct'}


In [61]:
other_document_schema = T.StructType.fromJson(
    json.loads(document_with_schema.schema.json())
)

other_document_schema == document_with_schema.schema

True

## **Explode and Collect**

In [67]:
document.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [65]:
episodes = document.select(
    'id', F.explode('_embedded.episodes').alias('episodes')
)

episodes.show(5, truncate=70)

+---+----------------------------------------------------------------------+
| id|                                                              episodes|
+---+----------------------------------------------------------------------+
|143|{{{http://api.tvmaze.com/episodes/10897}}, 2014-04-06, 2014-04-07T0...|
|143|{{{http://api.tvmaze.com/episodes/10898}}, 2014-04-13, 2014-04-14T0...|
|143|{{{http://api.tvmaze.com/episodes/10899}}, 2014-04-20, 2014-04-21T0...|
|143|{{{http://api.tvmaze.com/episodes/10900}}, 2014-04-27, 2014-04-28T0...|
|143|{{{http://api.tvmaze.com/episodes/10901}}, 2014-05-04, 2014-05-05T0...|
+---+----------------------------------------------------------------------+
only showing top 5 rows


In [66]:
episodes.count()

53

In [70]:
episode_name_id = document.select(
    F.map_from_arrays(
        F.col("_embedded.episodes.id"), F.col("_embedded.episodes.name")
    ).alias('name_id')
)

episode_name_id.show(1, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [71]:
episode_name_id = episode_name_id.select(
    F.posexplode('name_id').alias('position', 'id', 'name')
)

episode_name_id.show(5)

+--------+-----+--------------------+
|position|   id|                name|
+--------+-----+--------------------+
|       0|10897|Minimum Viable Pr...|
|       1|10898|       The Cap Table|
|       2|10899|Articles of Incor...|
|       3|10900|    Fiduciary Duties|
|       4|10901|      Signaling Risk|
+--------+-----+--------------------+
only showing top 5 rows


> Both **`explode()`** and **`posexplode()`** will skip any null values in the array or the map. If you want to have null as records, you can use **`explode_outer()`** or **`posexplode_outer()`** the same way.

In [74]:
episodes.show(5)

+---+--------------------+
| id|            episodes|
+---+--------------------+
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
+---+--------------------+
only showing top 5 rows


In [75]:
collected_episodes = episodes.groupby('id').agg(
    F.collect_list('episodes').alias('episodes')
)

collected_episodes.show()

+---+--------------------+
| id|            episodes|
+---+--------------------+
|143|[{{{http://api.tv...|
+---+--------------------+



In [76]:
collected_episodes.count()

1

In [77]:
collected_episodes.printSchema()

root
 |-- id: long (nullable = true)
 |-- episodes: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- _links: struct (nullable = true)
 |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |-- href: string (nullable = true)
 |    |    |-- airdate: string (nullable = true)
 |    |    |-- airstamp: string (nullable = true)
 |    |    |-- airtime: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- image: struct (nullable = true)
 |    |    |    |-- medium: string (nullable = true)
 |    |    |    |-- original: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- number: long (nullable = true)
 |    |    |-- runtime: long (nullable = true)
 |    |    |-- season: long (nullable = true)
 |    |    |-- summary: string (nullable = true)
 |    |    |-- url: string (nullable = true)



In [78]:
struct_ex = document.select(
    F.struct(
        F.col("status"), F.col("weight"), F.lit(True).alias("has_watched")
    ).alias('info')
)

In [79]:
struct_ex.show(1, False)

+-----------------+
|info             |
+-----------------+
|{Ended, 96, true}|
+-----------------+



In [80]:
struct_ex.printSchema()

root
 |-- info: struct (nullable = false)
 |    |-- status: string (nullable = true)
 |    |-- weight: long (nullable = true)
 |    |-- has_watched: boolean (nullable = false)



# **Additional exercises**

#### **Exercise 6.5**
Although much less common, you can create a data frame from a dictionary. Since dictionaries are so close to JSON documents, build the schema for ingesting the following dictionary. (Both JSON or PySpark schemas are valid here.)
```python
dict_schema = ???
spark.createDataFrame([{"one": 1, "two": [1,2,3]}], schema=dict_schema)
```

In [10]:
df = spark.createDataFrame(
    [
        {
            'one': 1,
            'two': [1, 2, 3, ]
        }
    ]
)

df.printSchema()

root
 |-- one: long (nullable = true)
 |-- two: array (nullable = true)
 |    |-- element: long (containsNull = true)



In [11]:
df.show()

+---+---------+
|one|      two|
+---+---------+
|  1|[1, 2, 3]|
+---+---------+



In [12]:
schema = T.StructType(
    [
        T.StructField('one', T.LongType()),
        T.StructField('two', T.ArrayType(T.LongType()))
    ]
)

df = spark.createDataFrame(
    [
        {
            'one': 1,
            'two': [1, 2, 3, ]
        }
    ],
    schema=schema
)


df.printSchema()

root
 |-- one: long (nullable = true)
 |-- two: array (nullable = true)
 |    |-- element: long (containsNull = true)



In [13]:
df.show()

+---+---------+
|one|      two|
+---+---------+
|  1|[1, 2, 3]|
+---+---------+



In [14]:
df.select(
    'one',
    F.explode('two').alias('two')
).show()

+---+---+
|one|two|
+---+---+
|  1|  1|
|  1|  2|
|  1|  3|
+---+---+



In [27]:
df.select(
    F.map_from_arrays(
        F.array(*[F.lit(i) for i in range(1, 4)]), 
        'two'
    )
).show(1, False)

+------------------------------------+
|map_from_arrays(array(1, 2, 3), two)|
+------------------------------------+
|{1 -> 1, 2 -> 2, 3 -> 3}            |
+------------------------------------+



# **Blending Python and SQL code**

**ANSI SQL**, **HiveQL**, and **Spark SQL** represent the evolution of SQL standards and how they are implemented across different data processing engines. While they look similar on the surface, they differ significantly in their compliance, execution engines, and behavioral strictness.

---

### **📊 Direct Comparison Matrix**

| Dimension | **ANSI SQL** | **HiveQL** | **Spark SQL** |
| --- | --- | --- | --- |
| **Nature** | A formal industry specification (ISO/IEC). | A proprietary dialect created for Apache Hive (Hadoop). | Spark’s distributed SQL engine and DataFrame API layer. |
| **Execution Engine** | None (implemented by databases like PostgreSQL, Snowflake). | MapReduce / Tez (historically) or Hive execution engine. | Catalyst Optimizer running on Apache Spark (in-memory, RDD-based). |
| **Standard Compliance** | The definition of the standard itself. | Loosely based on SQL; historically diverged with custom syntax and limitations. | Aiming for ANSI-SQL compliance, configurable via ANSI mode (`spark.sql.ansi.enabled`). |
| **Error Handling** | Strict (e.g., throws errors on overflow, invalid casts). | Lenient (often returns `NULL` silently on invalid operations). | Configurable (Lenient by default; strict when ANSI mode is enabled). |

---

### **1. ANSI SQL: The Universal Standard**

**ANSI SQL** is not a software product or execution engine; it is a **formal specification** maintained by the American National Standards Institute (ANSI) and ISO.

* **Purpose:** It defines the syntax rules, data types, and expected behaviors for relational database management systems (RDBMS).
* **Role in Ecosystem:** Databases and query engines (like PostgreSQL, Snowflake, and modern Spark) measure their compliance against ANSI standards (e.g., SQL-92, SQL:1999, SQL:2016) to ensure portability.

---

### **2. HiveQL: The Hadoop Legacy**

**HiveQL** (Hive Query Language) was created by Facebook for Apache Hive to bring SQL-like querying to massive datasets stored in HDFS when MapReduce was the primary compute engine.

* **Characteristics:** While it borrowed heavily from SQL syntax, early HiveQL diverged significantly from ANSI standards. It had severe limitations (e.g., limited subquery support, no window functions in early versions) and introduced proprietary clauses like `LATERAL VIEW EXPLODE` or custom `SERDE` definitions.
* **Error Handling Philosophy:** HiveQL was designed to be **fault-tolerant at all costs**. If a data type conversion failed or division by zero occurred, it would silently return `NULL` rather than failing the job, which often masked upstream data quality bugs.

---

### **3. Spark SQL: The Modern Distributed Engine**

**Spark SQL** is Apache Spark's module for working with structured data, supporting both standard SQL queries and the programmatic DataFrame API.

* **Compatibility Bridge:** Spark SQL originally inherited HiveQL syntax and metastore compatibility (`spark.enableHiveSupport()`) to make it easy for Hadoop/Hive engineers to migrate workloads to Spark.
* **The Catalyst Optimizer:** Unlike HiveQL (which mapped queries to rigid MapReduce jobs), Spark SQL compiles queries into optimized physical execution plans via the Catalyst Optimizer and Tungsten execution engine.
* **ANSI Mode:** Modern Spark includes an **ANSI compliance mode** (`spark.sql.ansi.enabled = true`). When enabled, Spark shifts from Hive-like leniency to strict ANSI SQL behavior:
* Throwing runtime exceptions on arithmetic overflows (instead of wrapping around).
* Rejecting invalid string-to-timestamp or numeric casts instead of silently returning `NULL`.

In [30]:
df_elements = spark.read.csv(
    "../data/elements/Periodic_Table_Of_Elements.csv",
    header=True,
    inferSchema=True
)

In [31]:
df_elements.where(F.col('phase') == 'liq').groupby('period').count().show()

+------+-----+
|period|count|
+------+-----+
|     6|    1|
|     4|    1|
+------+-----+



In [35]:
# spark.sql("""
# SELECT
#     period,
#     count(*)
# FROM df_elements
# WHERE phase = 'liq'
# GROUP BY period;
# """).show()

![](../images/PySpark_SQL.png)

In [36]:
try:
    spark.sql(
        """
        SELECT period, count(*)
        FROM df_elements
        WHERE phase='liq'
        GROUP BY period
        """
    ).show(5)
except AnalysisException as e:
    print(e)

{"ts": "2026-11-13 20:23:49.858", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `df_elements` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o26.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `df_elements` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with 

[TABLE_OR_VIEW_NOT_FOUND] The table or view `df_elements` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 3 pos 13;
'Aggregate ['period], ['period, unresolvedalias(count(1))]
+- 'Filter ('phase = liq)
   +- 'UnresolvedRelation [df_elements], [], false



In [37]:
df_elements.createOrReplaceTempView('elements')

26/11/13 20:28:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [38]:
spark.sql("""
SELECT period, count(*) AS count
from elements
WHERE phase='liq'
GROUP BY period
""").show()

+------+-----+
|period|count|
+------+-----+
|     6|    1|
|     4|    1|
+------+-----+



In [39]:
spark.catalog

In [40]:
spark.catalog.listTables()

[Table(name='elements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [41]:
spark.catalog.dropTempView('elements')

True

In [42]:
spark.catalog.listTables()

[]

## **SQL and Spark**

In [4]:
df = spark.read.csv(
    '../data/backblaze/data_Q3_2019',
    header=True,
    inferSchema=True
)

In [45]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- model: string (nullable = true)
 |-- capacity_bytes: long (nullable = true)
 |-- failure: integer (nullable = true)
 |-- smart_1_normalized: integer (nullable = true)
 |-- smart_1_raw: integer (nullable = true)
 |-- smart_2_normalized: integer (nullable = true)
 |-- smart_2_raw: integer (nullable = true)
 |-- smart_3_normalized: integer (nullable = true)
 |-- smart_3_raw: integer (nullable = true)
 |-- smart_4_normalized: integer (nullable = true)
 |-- smart_4_raw: integer (nullable = true)
 |-- smart_5_normalized: integer (nullable = true)
 |-- smart_5_raw: integer (nullable = true)
 |-- smart_7_normalized: integer (nullable = true)
 |-- smart_7_raw: long (nullable = true)
 |-- smart_8_normalized: integer (nullable = true)
 |-- smart_8_raw: integer (nullable = true)
 |-- smart_9_normalized: integer (nullable = true)
 |-- smart_9_raw: integer (nullable = true)
 |-- smart_10_normalized: integer (nul

In [46]:
df.columns

['date',
 'serial_number',
 'model',
 'capacity_bytes',
 'failure',
 'smart_1_normalized',
 'smart_1_raw',
 'smart_2_normalized',
 'smart_2_raw',
 'smart_3_normalized',
 'smart_3_raw',
 'smart_4_normalized',
 'smart_4_raw',
 'smart_5_normalized',
 'smart_5_raw',
 'smart_7_normalized',
 'smart_7_raw',
 'smart_8_normalized',
 'smart_8_raw',
 'smart_9_normalized',
 'smart_9_raw',
 'smart_10_normalized',
 'smart_10_raw',
 'smart_11_normalized',
 'smart_11_raw',
 'smart_12_normalized',
 'smart_12_raw',
 'smart_13_normalized',
 'smart_13_raw',
 'smart_15_normalized',
 'smart_15_raw',
 'smart_16_normalized',
 'smart_16_raw',
 'smart_17_normalized',
 'smart_17_raw',
 'smart_22_normalized',
 'smart_22_raw',
 'smart_23_normalized',
 'smart_23_raw',
 'smart_24_normalized',
 'smart_24_raw',
 'smart_168_normalized',
 'smart_168_raw',
 'smart_170_normalized',
 'smart_170_raw',
 'smart_173_normalized',
 'smart_173_raw',
 'smart_174_normalized',
 'smart_174_raw',
 'smart_177_normalized',
 'smart_177_r

In [47]:
df = df.select(
    [ F.col(x).cast(T.LongType()) if x.startswith('smart') else F.col(x)
        for x in df.columns
    ]
)

df.printSchema()

root
 |-- date: date (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- model: string (nullable = true)
 |-- capacity_bytes: long (nullable = true)
 |-- failure: integer (nullable = true)
 |-- smart_1_normalized: long (nullable = true)
 |-- smart_1_raw: long (nullable = true)
 |-- smart_2_normalized: long (nullable = true)
 |-- smart_2_raw: long (nullable = true)
 |-- smart_3_normalized: long (nullable = true)
 |-- smart_3_raw: long (nullable = true)
 |-- smart_4_normalized: long (nullable = true)
 |-- smart_4_raw: long (nullable = true)
 |-- smart_5_normalized: long (nullable = true)
 |-- smart_5_raw: long (nullable = true)
 |-- smart_7_normalized: long (nullable = true)
 |-- smart_7_raw: long (nullable = true)
 |-- smart_8_normalized: long (nullable = true)
 |-- smart_8_raw: long (nullable = true)
 |-- smart_9_normalized: long (nullable = true)
 |-- smart_9_raw: long (nullable = true)
 |-- smart_10_normalized: long (nullable = true)
 |-- smart_10_raw: long (nullable 

In [48]:
df.createOrReplaceTempView('backblaze_stats_2019')

In [53]:
spark.sql("""
SELECT serial_number
FROM backblaze_stats_2019
WHERE failure = 1
""").show(5, False)

+-------------+
|serial_number|
+-------------+
|ZA10MCJ5     |
|ZCH07T9K     |
|ZCH0CA7Z     |
|Z302F381     |
|ZCH0B3Z2     |
+-------------+
only showing top 5 rows


In [55]:
df.select(
    F.col('serial_number')
).where(F.col('failure') == 1).show(5, False)

+-------------+
|serial_number|
+-------------+
|ZA10MCJ5     |
|ZCH07T9K     |
|ZCH0CA7Z     |
|Z302F381     |
|ZCH0B3Z2     |
+-------------+
only showing top 5 rows


In [57]:
spark.sql("""
SELECT model, min(capacity_bytes / pow(1024, 3)) AS min_GB, max(capacity_bytes / pow(1024, 3)) AS max_GB
FROM backblaze_stats_2019
GROUP BY 1
ORDER BY 3 DESC
""").show(5, False)

[Stage 49:====================================================>   (29 + 2) / 31]

+--------------------+----------------------+-------+
|model               |min_GB                |max_GB |
+--------------------+----------------------+-------+
|TOSHIBA MG07ACA14TA |13039.0               |13039.0|
|ST12000NM0007       |-9.313225746154785E-10|11176.0|
|HGST HUH721212ALE600|11176.0               |11176.0|
|ST12000NM0117       |11176.0               |11176.0|
|HGST HUH721212ALN604|-9.313225746154785E-10|11176.0|
+--------------------+----------------------+-------+
only showing top 5 rows


In [58]:
df.groupby(F.col('model')).agg(
    F.min(F.col('capacity_bytes') / F.pow(F.lit(1024), 3)).alias('min_GB'),
    F.max(F.col('capacity_bytes') / F.pow(F.lit(1024), 3)).alias('max_GB')
).orderBy(F.col('max_GB'), ascending=False).show(5)

[Stage 52:====================================================>   (29 + 2) / 31]

+--------------------+--------------------+-------+
|               model|              min_GB| max_GB|
+--------------------+--------------------+-------+
| TOSHIBA MG07ACA14TA|             13039.0|13039.0|
|       ST12000NM0007|-9.31322574615478...|11176.0|
|HGST HUH721212ALE600|             11176.0|11176.0|
|       ST12000NM0117|             11176.0|11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|11176.0|
+--------------------+--------------------+-------+
only showing top 5 rows


In [59]:
spark.sql("""
SELECT model, min(capacity_bytes / pow(1024, 3)) min_GB, max(capacity_bytes / pow(1024, 3)) max_GB
FROM backblaze_stats_2019
GROUP BY 1
HAVING min_GB != max_GB
ORDER BY 3 DESC
""").show(5)

[Stage 55:====================================================>   (29 + 2) / 31]

+--------------------+--------------------+-----------------+
|               model|              min_GB|           max_GB|
+--------------------+--------------------+-----------------+
|       ST12000NM0007|-9.31322574615478...|          11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|          11176.0|
|HGST HUH721010ALE600|-9.31322574615478...|           9314.0|
|       ST10000NM0086|-9.31322574615478...|           9314.0|
|        ST8000NM0055|-9.31322574615478...|7452.036460876465|
+--------------------+--------------------+-----------------+
only showing top 5 rows


In [62]:
df.groupby(F.col('model')).agg(
    F.min(F.col('capacity_bytes') / F.pow(F.lit(1024), 3)).alias('min_GB'),
    F.max(F.col('capacity_bytes') / F.pow(F.lit(1024), 3)).alias('max_GB')
).where(F.col('min_GB') != F.col('max_GB')).orderBy(
    F.col('max_GB'), ascending=False
).show(5)

[Stage 61:====================================================>   (29 + 2) / 31]

+--------------------+--------------------+-----------------+
|               model|              min_GB|           max_GB|
+--------------------+--------------------+-----------------+
|       ST12000NM0007|-9.31322574615478...|          11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|          11176.0|
|HGST HUH721010ALE600|-9.31322574615478...|           9314.0|
|       ST10000NM0086|-9.31322574615478...|           9314.0|
|        ST8000NM0055|-9.31322574615478...|7452.036460876465|
+--------------------+--------------------+-----------------+
only showing top 5 rows


In [5]:
df.createOrReplaceTempView('drive_stats')

26/11/14 14:45:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [6]:
spark.catalog.listTables()

[Table(name='drive_stats', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [9]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW drive_days AS
SELECT model, COUNT(*) as drive_days
FROM drive_stats
GROUP BY model
""")

DataFrame[]

In [10]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW failures AS
SELECT model, COUNT(*) AS failures
FROM drive_stats
WHERE failure = 1
GROUP BY model
""")

DataFrame[]

In [13]:
drive_days = df.groupby(F.col('model')).agg(
    F.count(F.col('*')).alias('drive_days')
)

In [15]:
failures = (
    df.where(F.col('failure') == 1)
    .groupby(F.col('model'))
    .agg(
        F.count(F.col('*')).alias('failures')
    )
)

> **Creating tables from data in SQL**
You can also create a table from data on a hard drive or HDFS. For this, you can use a modified SQL query. Since we are reading a CSV file, we prefix our path with csv.:
```python
    spark.sql("create table q1 as select * from
    csv.`./data/backblaze/drive_stats_2019_Q1`")
```

> **PySpark’s union() vs. SQL UNION**
In SQL, `UNION` removes the duplicate records. PySpark’s `union()` doesn’t, which is why it’s equivalent to a SQL `UNION ALL`. If you want to drop the duplicates, which is an expensive operation when working in a distributed context, use the `distinct()` function after your `union()`. This is one of the rare cases where PySpark’s vocabulary doesn’t follow SQL’s, but it’s for a good reason. Most of the time, you’ll want the `UNION ALL` behavior.

In [16]:
spark.sql("""
SELECT DD.model, drive_days, failures
FROM drive_days as DD
LEFT JOIN failures F
ON DD.model = F.model
""").show(5)

[Stage 12:====================================================>   (29 + 2) / 31]

+-------------+----------+--------+
|        model|drive_days|failures|
+-------------+----------+--------+
|  ST9250315AS|        89|    NULL|
|  ST4000DM000|   1796728|      72|
|ST12000NM0007|   3212635|     361|
|  ST8000DM005|      2280|       1|
|   ST320LT007|        89|    NULL|
+-------------+----------+--------+
only showing top 5 rows


In [17]:
drive_days.join(
    failures, on='model', how='left'
).show(5)

[Stage 18:====================================================>   (29 + 2) / 31]

+-------------+----------+--------+
|        model|drive_days|failures|
+-------------+----------+--------+
|  ST9250315AS|        89|    NULL|
|  ST4000DM000|   1796728|      72|
|ST12000NM0007|   3212635|     361|
|  ST8000DM005|      2280|       1|
|   ST320LT007|        89|    NULL|
+-------------+----------+--------+
only showing top 5 rows


In [18]:
spark.sql("""
SELECT failures.model, failures / drive_days failure_rate
FROM (
    SELECT model, count(*) AS drive_days
    FROM drive_stats
    GROUP BY model) drive_days
INNER JOIN (
    SELECT model, count(*) AS failures
    FROM drive_stats
    WHERE failure = 1
    GROUP BY model) failures
ON
drive_days.model = failures.model
ORDER BY 2 desc
"""
).show(5)

[Stage 24:====================================================>   (29 + 2) / 31]

+------------------+--------------------+
|             model|        failure_rate|
+------------------+--------------------+
|     ST12000NM0117|0.019305019305019305|
|TOSHIBA MQ01ABF050|5.579360828423496E-4|
|       ST8000DM005|4.385964912280702E-4|
|        ST500LM030| 4.19639110365086E-4|
|     ST500LM012 HN|1.511585221015353...|
+------------------+--------------------+
only showing top 5 rows


In [19]:
spark.sql("""
WITH drive_days as (
    SELECT model, count(*) AS drive_days
    FROM drive_stats
    GROUP BY model),
failures as (
    SELECT model, count(*) AS failures
    FROM drive_stats
    WHERE failure = 1
    GROUP BY model)
SELECT failures.model, failures / drive_days failure_rate
FROM drive_days
INNER JOIN failures
ON
drive_days.model = failures.model
ORDER BY 2 desc
""").show(5)

[Stage 30:====================================================>   (29 + 2) / 31]

+------------------+--------------------+
|             model|        failure_rate|
+------------------+--------------------+
|     ST12000NM0117|0.019305019305019305|
|TOSHIBA MQ01ABF050|5.579360828423496E-4|
|       ST8000DM005|4.385964912280702E-4|
|        ST500LM030| 4.19639110365086E-4|
|     ST500LM012 HN|1.511585221015353...|
+------------------+--------------------+
only showing top 5 rows


In [21]:
def failure_rate(drive_stats):
    drive_days = drive_stats.groupby(F.col("model")).agg(
        F.count(F.col("*")).alias("drive_days")
    )

    failures = (
        drive_stats.where(F.col("failure") == 1)
            .groupby(F.col("model"))
            .agg(F.count(F.col("*")).alias("failures"))
    )

    answer = (
        drive_days.join(failures, on="model", how="inner")
            .withColumn("failure_rate", F.col("failures") / F.col("drive_days"))
            .orderBy(F.col("failure_rate").desc())
    )

    return answer

In [22]:
failure_rate(df).show(5)

[Stage 36:====================================================>   (29 + 2) / 31]

+------------------+----------+--------+--------------------+
|             model|drive_days|failures|        failure_rate|
+------------------+----------+--------+--------------------+
|     ST12000NM0117|       259|       5|0.019305019305019305|
|TOSHIBA MQ01ABF050|     44808|      25|5.579360828423496E-4|
|       ST8000DM005|      2280|       1|4.385964912280702E-4|
|        ST500LM030|     21447|       9| 4.19639110365086E-4|
|     ST500LM012 HN|     46309|       7|1.511585221015353...|
+------------------+----------+--------+--------------------+
only showing top 5 rows


In [24]:
print("drive_days" in dir())

True


### **Data Ingestion**

In [26]:
from functools import reduce

In [27]:
from functools import reduce

# 1. Define the directory path where the dataset files are stored
DATA_DIRECTORY = "../data/backblaze/"

# 2. Define a list of file/folder names to process (e.g., Backblaze hard drive stats for Q3 2019)
DATA_FILES = [
    "data_Q3_2019",
]

# 3. Read each CSV file into a list of PySpark DataFrames.
#    - header=True: Treats the first row as column names.
#    - inferSchema=True: Automatically detects data types (e.g., int, string, date).
data = [
    spark.read.csv(DATA_DIRECTORY + file, header=True, inferSchema=True)
    for file in DATA_FILES
]

# 4. Find the intersection of column names across all loaded DataFrames.
#    - [set(df.columns) for df in data]: Converts each DataFrame's columns into a set.
#    - reduce(lambda x, y: x.intersection(y), ...): Computes the common columns
#      that exist across ALL DataFrames in the list.
common_columns = list(
    reduce(lambda x, y: x.intersection(y), [set(df.columns) for df in data])
)

# 5. Validate that crucial columns required for downstream analysis are present.
#    - If "model", "capacity_bytes", "date", or "failure" are missing from common_columns,
#      Python throws an AssertionError, halting execution early before bad data propagates.
assert set(["model", "capacity_bytes", "date", "failure"]).issubset(
    set(common_columns)
)

# 6. Unify/combine all DataFrames into a single master DataFrame.
#    - .select(common_columns): Ensures every DataFrame only selects columns that are
#      shared across all datasets (preventing schema mismatch errors during union).
#    - reduce(lambda x, y: x.union(..., data)): Sequentially stacks (appends) all DataFrames
#      together into one large unified DataFrame (`full_data`).
full_data = reduce(
    lambda x, y: x.select(common_columns).union(y.select(common_columns)), data
)

#### **🔑 Key Engineering Patterns Used:**

* **`reduce()`** from **`functools`**: Used to iteratively fold a list of items down into a single result—first used here to find the global intersection of columns, and second to chain `.union()` operations across multiple DataFrames.
* **Defensive Assertion (`assert`):** Protects data pipelines from schema drift by crashing early if essential operational columns (like `failure` or `date`) go missing.
* **Schema Alignment via `.select(common_columns)`:** Vital when working with real-world append-heavy datasets (like quarterly log files where column orders or optional columns might change slightly over time) to guarantee that `.union()` operations never fail due to misplaced or missing columns.

### **Data Processing**

In [28]:
# 1. Standardize and transform columns using SQL expressions (.selectExpr)
full_data = full_data.selectExpr(
    "model",
    # Convert capacity from raw bytes to Gigabytes (GB) by dividing by 1024^3
    "capacity_bytes / pow(1024, 3) capacity_GB",
    "date",
    "failure",
)

# 2. Calculate total "drive days" (exposure time) for each model and capacity
#    - Each row typically represents 1 day of operation for a single hard drive.
#    - Counting total rows per group tells us how many total days drives of that model operated.
drive_days = full_data.groupby("model", "capacity_GB").agg(
    F.count("*").alias("drive_days")
)

# 3. Calculate total failures for each model and capacity
#    - .where("failure = 1"): Filters only the records where a drive failure occurred.
#    - Aggregates the count of failed days per model and capacity.
failures = (
    full_data.where("failure = 1")
    .groupby("model", "capacity_GB")
    .agg(F.count("*").alias("failures"))
)

# 4. Combine metrics and compute the final failure rate
summarized_data = (
    # Join total drive days with failure counts using a left join
    # (preserving all models even if they had zero failures)
    drive_days.join(failures, on=["model", "capacity_GB"], how="left")
    # Fill null values in the 'failures' column with 0.0 (for models that never failed)
    .fillna(0.0, ["failures"])
    # Calculate annualized or operational failure rate (failures / total operational days)
    .selectExpr("model", "capacity_GB", "failures / drive_days failure_rate")
    # .cache(): Persists the final summarized DataFrame in memory/disk
    # to speed up repeated downstream actions or iterations.
    .cache()
)

#### **🔑 Key Engineering Patterns Used:**

* **`.selectExpr()`:** Allows you to write inline SQL expressions (like `capacity_bytes / pow(1024, 3)`) and assign column aliases directly within a transformation string without importing math functions.
* **Left Join + `.fillna()` Pattern:** Essential when calculating rates or ratios. If a drive model experienced zero failures, the inner or left-join would produce a `NULL` for the failure count; filling it with `0.0` prevents subsequent mathematical operations from resulting in `NULL` or NaN outcomes.
* **`.cache()` Optimization:** Used here because `summarized_data` is likely going to be queried, plotted, or written out multiple times. Caching prevents Spark from re-computing the entire upstream CSV reading and aggregation pipeline from scratch on every action.

In [29]:
def most_reliable_drive_for_capacity(
    data, capacity_GB=2048, precision=0.25, top_n=3
):
    """Returns the top 3 drives for a given approximate capacity.
    Given a capacity in GB and a precision as a decimal number, we keep the N
    drives where:
    - the capacity is between (capacity * 1/(1+precision)), capacity *
    (1+precision)
    - the failure rate is the lowest
    """
    # 1. Calculate the lower bound of the acceptable capacity range
    #    - Example: for 2048 GB with 0.25 precision, 2048 / 1.25 = 1638.4 GB
    capacity_min = capacity_GB / (1 + precision)

    # 2. Calculate the upper bound of the acceptable capacity range
    #    - Example: for 2048 GB with 0.25 precision, 2048 * 1.25 = 2560.0 GB
    capacity_max = capacity_GB * (1 + precision)

    # 3. Filter and rank the data using Spark DataFrame transformations
    answer = (
        data
        # Filter rows where 'capacity_GB' falls within our defined tolerance range
        .where(f"capacity_GB between {capacity_min} and {capacity_max}")
        # Sort the filtered results:
        # - Primary sort: 'failure_rate' ascending (lowest failure rate comes first)
        # - Secondary sort: 'capacity_GB' descending (highest capacity first if rates match)
        .orderBy("failure_rate", "capacity_GB", ascending=[True, False])
        # Keep only the top N rows (default is top 3)
        .limit(top_n)
    )

    # 4. Return the resulting DataFrame containing the most reliable drives
    return answer

#### **🔑 Key Engineering Patterns Used:**

* **Dynamic Range Filtering (`between ... and ...`):** Instead of looking for an exact hard drive size match (which is rare due to slight sizing variations across manufacturers), it creates a flexible tolerance window using the `precision` parameter.
* **Multi-Column Sorting with Direction Arrays (`ascending=[True, False]`):** When sorting by multiple columns in PySpark, passing a list to `ascending` allows you to specify a different sorting direction for each column—sorting `failure_rate` in ascending order (best first) while breaking ties by sorting `capacity_GB` in descending order (largest first).

In [30]:
most_reliable_drive_for_capacity(summarized_data, capacity_GB=11176.0).show()

[Stage 55:=====================================================>(199 + 1) / 200]

+--------------------+-----------+--------------------+
|               model|capacity_GB|        failure_rate|
+--------------------+-----------+--------------------+
|HGST HUH721010ALE600|     9314.0|                 0.0|
|HGST HUH721212ALN604|    11176.0|1.585588480593982...|
|HGST HUH721212ALE600|    11176.0|1.636661211129296...|
+--------------------+-----------+--------------------+



# **Additional exercises**